# AutoValue — Model v002 Error & Residual Analysis

This notebook provides empirical residual distributions, error breakdown across vehicle segments, and performance across price, age, and mileage bands for AutoValue Model v002.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from data_pipeline.feature_engineering.features import CarFeatureEngineer
from ml.features.preprocessor import NUMERICAL_FEATURES, CATEGORICAL_FEATURES
from ml.training.split import duplicate_safe_train_test_split
from ml.evaluation.error_analysis import ErrorAnalyzer

# Load processed data
df = pd.read_csv('../data/processed/autovalue_processed.csv')
engineer = CarFeatureEngineer()
eng_df = engineer.transform(df)

# Load model and preprocessor v002
model = joblib.load('../models/model_v002.joblib')
prep = joblib.load('../models/preprocessor_v002.joblib')
metadata = json.load(open('../models/metadata_v002.json'))

train_df, test_df = duplicate_safe_train_test_split(eng_df, test_size=0.20, random_state=42)
X_test = test_df[NUMERICAL_FEATURES + CATEGORICAL_FEATURES]
y_test = test_df['listed_price']

X_test_prep = prep.transform(X_test)
y_pred = model.predict(X_test_prep)

analyzer = ErrorAnalyzer()
error_report = analyzer.run_full_error_analysis(y_test.values, y_pred, test_df)
res_df = error_report['residuals_dataframe']
print('Overall Test MAE:', error_report['overall_metrics']['mae'])
print('Overall Test R2:', metadata['metrics_test']['r2'])

In [ ]:
# Residual distribution plot
plt.figure(figsize=(10, 5))
sns.histplot(res_df['residual'], kde=True, bins=25, color='royalblue')
plt.title('Residual Distribution (Actual - Predicted) - Model v002')
plt.xlabel('Residual (INR)')
plt.ylabel('Count')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Actual vs Predicted Scatter
plt.figure(figsize=(8, 8))
plt.scatter(res_df['actual_price']/1e5, res_df['predicted_price']/1e5, alpha=0.7, color='teal')
plt.plot([0, 45], [0, 45], 'r--', label='Perfect Prediction (y=x)')
plt.title('Actual vs Predicted Listing Price (Lakh INR)')
plt.xlabel('Actual Price (₹ Lakh)')
plt.ylabel('Predicted Price (₹ Lakh)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()